In [1]:
import pandas as pd
# Load the file to inspect its structure
posts = pd.read_csv("posts_table.csv")
posts.head()

,post_id,user_id,full_text,created_at,lang,repost_count,like_count,source
0,1907578844636393679,1810361594586738692,"This is nasty \n\nNASDAQ futures down -4.4%, S...",Wed Apr 02 23:40:12 +0000 2025,en,0,0,Twitter Web App
1,1907578044492968060,2736639061,Investors Panic as U.S. Stock Market Plummets ...,Wed Apr 02 23:37:01 +0000 2025,en,1,0,Twitter Web App
2,1907575218501198232,1850383929829945344,For all the people who have absolutely no conc...,Wed Apr 02 23:25:47 +0000 2025,en,0,2,Twitter for Android
3,1907574739964510255,449290925,As someone who is feeling JVL levels of schade...,Wed Apr 02 23:23:53 +0000 2025,en,0,0,Twitter for Android
4,1907570533836701721,23709151,The fuzziest math is #TrumpTariff Math.,Wed Apr 02 23:07:10 +0000 2025,en,0,0,Twitter for Android


In [2]:
import re

def clean_text_preprocessing1(text):
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)     # Remove URLs
    text = re.sub(r"@\w+", "", text)                        # Remove mentions
    text = re.sub(r"#", "", text)                           # Remove only '#' 
    text = re.sub(r"[^\x00-\x7F]+", "", text)               # Remove emojis (non-ASCII characters)
    text = re.sub(r"\d+", "", text)                         # Remove numerical characters
    return text

# Apply this to DataFrame 
posts['text_preprocessing1'] = posts['full_text'].apply(clean_text_preprocessing1)
posts['text_preprocessing1'].head()

0    This is nasty \n\nNASDAQ futures down -.%, S&a...
1    Investors Panic as U.S. Stock Market Plummets ...
2    For all the people who have absolutely no conc...
3    As someone who is feeling JVL levels of schade...
4               The fuzziest math is TrumpTariff Math.
Name: text_preprocessing1, dtype: object

In [3]:
import string

def clean_text_preprocessing2(text):
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)  # Remove punctuation
    return text

# Apply this to the result of step 1
posts['text_preprocessing_no_punctuation'] = posts['text_preprocessing1'].apply(clean_text_preprocessing2)
posts['text_preprocessing_no_punctuation'].head()

0    This is nasty \n\nNASDAQ futures down  SampP f...
1    Investors Panic as US Stock Market Plummets Ov...
2    For all the people who have absolutely no conc...
3    As someone who is feeling JVL levels of schade...
4                The fuzziest math is TrumpTariff Math
Name: text_preprocessing_no_punctuation, dtype: object

In [4]:
# Convert texts to lowercase
posts['text_preprocessing_no_punctuation'] = \
posts['text_preprocessing_no_punctuation'].map(lambda x: x.lower())
posts['text_preprocessing_no_punctuation'].head()

0    this is nasty \n\nnasdaq futures down  sampp f...
1    investors panic as us stock market plummets ov...
2    for all the people who have absolutely no conc...
3    as someone who is feeling jvl levels of schade...
4                the fuzziest math is trumptariff math
Name: text_preprocessing_no_punctuation, dtype: object

In [5]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def remove_stopwords(text):
    words = text.split()
    filtered = [word for word in words if word not in ENGLISH_STOP_WORDS]
    return " ".join(filtered)

# Apply to cleaned column
posts['text_no_stopwords'] = posts['text_preprocessing_no_punctuation'].apply(remove_stopwords)
posts['text_no_stopwords'].head()

0    nasty nasdaq futures sampp futures tariff news...
1    investors panic stock market plummets trumps t...
2    people absolutely concept tariff amp differs g...
3    feeling jvl levels schadenfreude let just say ...
4                       fuzziest math trumptariff math
Name: text_no_stopwords, dtype: object

In [6]:
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk import pos_tag

# Download required resources
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')

lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN
        
def lemmatize_with_pos(text):
    tokens = nltk.word_tokenize(text)
    tagged_tokens = pos_tag(tokens)
    lemmatized = [
        lemmatizer.lemmatize(word, get_wordnet_pos(pos)) 
        for word, pos in tagged_tokens
    ]
    return " ".join(lemmatized)
    
posts['text_lemmatized'] = posts['text_no_stopwords'].apply(lemmatize_with_pos)
posts['text_lemmatized'].head()

[nltk_data] Downloading package punkt to /Users/patnbe/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/patnbe/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/patnbe/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/patnbe/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


0    nasty nasdaq future sampp future tariff news h...
1    investor panic stock market plummet trump tari...
2    people absolutely concept tariff amp differs g...
3    feel jvl level schadenfreude let just say happ...
4                          fuzzy math trumptariff math
Name: text_lemmatized, dtype: object

In [7]:
posts.head()

,post_id,user_id,full_text,created_at,lang,repost_count,like_count,source,text_preprocessing1,text_preprocessing_no_punctuation,text_no_stopwords,text_lemmatized
0,1907578844636393679,1810361594586738692,"This is nasty \n\nNASDAQ futures down -4.4%, S...",Wed Apr 02 23:40:12 +0000 2025,en,0,0,Twitter Web App,"This is nasty \n\nNASDAQ futures down -.%, S&a...",this is nasty \n\nnasdaq futures down sampp f...,nasty nasdaq futures sampp futures tariff news...,nasty nasdaq future sampp future tariff news h...
1,1907578044492968060,2736639061,Investors Panic as U.S. Stock Market Plummets ...,Wed Apr 02 23:37:01 +0000 2025,en,1,0,Twitter Web App,Investors Panic as U.S. Stock Market Plummets ...,investors panic as us stock market plummets ov...,investors panic stock market plummets trumps t...,investor panic stock market plummet trump tari...
2,1907575218501198232,1850383929829945344,For all the people who have absolutely no conc...,Wed Apr 02 23:25:47 +0000 2025,en,0,2,Twitter for Android,For all the people who have absolutely no conc...,for all the people who have absolutely no conc...,people absolutely concept tariff amp differs g...,people absolutely concept tariff amp differs g...
3,1907574739964510255,449290925,As someone who is feeling JVL levels of schade...,Wed Apr 02 23:23:53 +0000 2025,en,0,0,Twitter for Android,As someone who is feeling JVL levels of schade...,as someone who is feeling jvl levels of schade...,feeling jvl levels schadenfreude let just say ...,feel jvl level schadenfreude let just say happ...
4,1907570533836701721,23709151,The fuzziest math is #TrumpTariff Math.,Wed Apr 02 23:07:10 +0000 2025,en,0,0,Twitter for Android,The fuzziest math is TrumpTariff Math.,the fuzziest math is trumptariff math,fuzziest math trumptariff math,fuzzy math trumptariff math


In [8]:
# Export cleaned_text.csv file for using in BERTopic model that is performed in another environment due to the conflict in versions of some libraries used in both models
posts.to_csv('cleaned_text.csv', index=False)

In [9]:
# Tokenize text
tokenized_text = posts['text_lemmatized'].apply(lambda x: x.split())

In [10]:
from gensim import corpora

# Create Dictionary
id2word = corpora.Dictionary(tokenized_text)

# Create Corpus: Term Document Frequency
corpus = [id2word.doc2bow(text) for text in tokenized_text]

In [33]:
from gensim.models.ldamodel import LdaModel

# Train LDA model
lda_model = LdaModel(
    corpus=corpus,
    id2word=id2word,
    num_topics=5, # number of topics
    random_state=100,
    update_every=1,
    chunksize=100,
    passes=10,
    alpha='auto',
    per_word_topics=True
)

In [26]:
# Display topics (word distribution in each topic)
topics = lda_model.print_topics()
for idx, topic in topics:
    print(f"Topic {idx + 1}: {topic}")

Topic 1: 0.059*"rate" + 0.032*"july" + 0.017*"state" + 0.017*"quota" + 0.016*"carney" + 0.016*"set" + 0.014*"help" + 0.013*"growth" + 0.012*"send" + 0.012*"sign"
Topic 2: 0.053*"like" + 0.025*"big" + 0.022*"right" + 0.018*"tell" + 0.017*"look" + 0.016*"job" + 0.016*"b" + 0.016*"uncertainty" + 0.012*"wrong" + 0.011*"lot"
Topic 3: 0.118*"tariff" + 0.068*"trump" + 0.016*"trade" + 0.012*"deal" + 0.011*"say" + 0.011*"war" + 0.010*"just" + 0.010*"country" + 0.009*"do" + 0.009*"amp"
Topic 4: 0.049*"market" + 0.019*"stock" + 0.018*"month" + 0.018*"he" + 0.016*"dollar" + 0.015*"im" + 0.013*"negotiation" + 0.012*"trillion" + 0.012*"lose" + 0.012*"pause"
Topic 5: 0.043*"canada" + 0.040*"tax" + 0.037*"pay" + 0.035*"china" + 0.024*"price" + 0.023*"good" + 0.020*"american" + 0.016*"import" + 0.015*"cost" + 0.014*"business"


In [27]:
# Display topic distribution in each document
lda_model.get_document_topics(corpus)
for i, doc in enumerate(corpus):
    topics = lda_model.get_document_topics(doc)
    print(f"Document {i + 1} Topics: {topics}")

Document 1 Topics: [(0, 0.041822787), (1, 0.062399764), (2, 0.63143444), (3, 0.1503861), (4, 0.113956906)]
Document 2 Topics: [(0, 0.05575785), (1, 0.055793434), (2, 0.613032), (3, 0.15524337), (4, 0.12017332)]
Document 3 Topics: [(0, 0.078425996), (1, 0.052410904), (2, 0.6810659), (3, 0.06494906), (4, 0.1231481)]
Document 4 Topics: [(0, 0.06000134), (1, 0.11877154), (2, 0.60949284), (3, 0.07325762), (4, 0.13847664)]
Document 5 Topics: [(0, 0.06841443), (1, 0.1332278), (2, 0.5562519), (3, 0.083699994), (4, 0.15840589)]
Document 6 Topics: [(0, 0.060377546), (1, 0.19846772), (2, 0.45377937), (3, 0.105858065), (4, 0.18151733)]
Document 7 Topics: [(0, 0.13080464), (1, 0.15697755), (2, 0.5202476), (3, 0.0663411), (4, 0.12562916)]
Document 8 Topics: [(0, 0.057198107), (1, 0.056497354), (2, 0.5201683), (3, 0.23344383), (4, 0.13269235)]
Document 9 Topics: [(0, 0.053785924), (1, 0.051253874), (2, 0.6856232), (3, 0.06378889), (4, 0.1455481)]
Document 10 Topics: [(0, 0.08928684), (1, 0.13518843),

In [ ]:
from gensim.models.coherencemodel import CoherenceModel

# Compute c_v coherence score
coherence_model_lda = CoherenceModel(model=lda_model, texts=tokenized_text, dictionary=id2word, coherence='c_v')
coherence_lda = coherence_model_lda.get_coherence()

print(f"C_v coherence score: {coherence_lda:.4f}")  # 5 topics =0.3110, 8 topics = 0.3048, 10 topics = 0.2684

C_v coherence score: 0.3110


In [30]:
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

# Prepare the visualisation
lda_vis = gensimvis.prepare(lda_model, corpus, id2word)
pyLDAvis.show(lda_vis, local=False, open_browser=True)

/opt/anaconda3/lib/python3.12/site-packages/joblib/externals/loky/backend/fork_exec.py:38: DeprecationWarning: This process (pid=35937) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()
/opt/anaconda3/lib/python3.12/site-packages/joblib/externals/loky/backend/fork_exec.py:38: DeprecationWarning: This process (pid=35937) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()
/opt/anaconda3/lib/python3.12/site-packages/joblib/externals/loky/backend/fork_exec.py:38: DeprecationWarning: This process (pid=35937) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()
/opt/anaconda3/lib/python3.12/site-packages/joblib/externals/loky/backend/fork_exec.py:38: DeprecationWarning: This process (pid=35937) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()
/opt/anaconda3/lib/python3.12/site-packages/joblib/externals/loky/backend/fork_exec.py:38: DeprecationWarnin

Serving to http://127.0.0.1:8889/    [Ctrl-C to exit]


127.0.0.1 - - [26/Aug/2025 04:27:34] "GET / HTTP/1.1" 200 -



stopping Server...
